# Salary Prediction — Analytics Report & Model Validation (VM)

Replaces the Streamlit dashboard (PLAN.md §17, removed 2026-08-18 - it hit repeated environment issues and a notebook fits this VM's constraints better anyway). Two parts:

- **Part A** — descriptive analytics graphs from `salary_analytics` (PLAN.md §15): avg salary by country/role/experience, salary distribution, technology popularity.
- **Part B** — takes real historical events from `developer_events` (which carry the real `ConvertedCompYearly`), runs them through the trained model directly, and compares predicted vs. real salary side by side.

**This notebook needs a Spark session with the Kafka connector** — launch Jupyter via `scripts/start_kafka_jupyter.sh` first.

**Prerequisites:** for Part A, `notebooks/run_dataset_producer.ipynb` and `notebooks/run_analytics_stream.ipynb` should have been run so `salary_analytics` has data. For Part B, just `notebooks/run_dataset_producer.ipynb` (Part B reads `developer_events` directly and scores it with the model itself — it does **not** need `run_prediction_stream.ipynb` to be running).

## 1. Path and working directory

In [ ]:
import sys, os

PROJECT_ROOT = "/home/linuxu/project"  # adjust if this VM's checkout lives elsewhere

sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print("Working directory:", os.getcwd())

## 1a. Install missing Python packages (one-time)

In [ ]:
%pip install python-dotenv matplotlib pandas

## 2. `.env` check

In [ ]:
if not os.path.exists(".env"):
    import subprocess
    subprocess.run(["cp", ".env.example", ".env"])
    print("Created .env from .env.example.")

print(open(".env").read())

## 3. Spark session — reuse the existing one (Kafka already works there)

In [ ]:
from src.common.spark_session import get_spark_session
from config import settings

spark = get_spark_session(app_name="SalaryAnalyticsReport", with_kafka=False)
spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)
print("MODEL_PATH exists?", os.path.exists(settings.MODEL_PATH))

## 3a. Preflight check: is the Kafka connector actually available?

If this cell fails, **stop here**: close this tab, run `scripts/start_kafka_jupyter.sh` in a terminal, and reopen this notebook from the NEW server it starts.

In [ ]:
try:
    (
        spark.readStream.format("kafka")
        .option("kafka.bootstrap.servers", settings.KAFKA_BOOTSTRAP_SERVERS)
        .option("subscribe", settings.KAFKA_DATASET_TOPIC)
        .load()
    )
    print("Kafka connector is available in this session - safe to proceed.")
except Exception as exc:
    if "Failed to find data source: kafka" in str(exc):
        print("Kafka connector NOT available. Close this tab, run scripts/start_kafka_jupyter.sh,")
        print("and reopen this notebook from the NEW server it starts.")
        raise RuntimeError("Kafka connector not available.") from None
    raise

## Part A — Descriptive Analytics (from `salary_analytics`)

In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd
from pyspark.sql import functions as F

results = (
    spark.read.format("kafka")
    .option("kafka.bootstrap.servers", settings.KAFKA_BOOTSTRAP_SERVERS)
    .option("subscribe", settings.KAFKA_ANALYTICS_TOPIC)
    .option("startingOffsets", "earliest")
    .load()
    .select(F.col("value").cast("string").alias("value"))
)
records = [json.loads(row["value"]) for row in results.collect()]
print(f"Total analytics records: {len(records)}")

by_metric = {}
for record in records:
    by_metric.setdefault(record["metric"], []).append(record)

event_counts_df = pd.DataFrame(by_metric.get("event_counts", []))
salary_breakdown_df = pd.DataFrame(by_metric.get("salary_breakdown", []))
technology_counts_df = pd.DataFrame(by_metric.get("technology_counts", []))

print(
    "event_counts:", len(event_counts_df),
    "| salary_breakdown:", len(salary_breakdown_df),
    "| technology_counts:", len(technology_counts_df),
)
if records and salary_breakdown_df.empty and technology_counts_df.empty:
    print("Got records but nothing usable - re-run run_dataset_producer.ipynb with a bigger LIMIT.")
elif not records:
    print("No analytics records yet. Run run_dataset_producer.ipynb and run_analytics_stream.ipynb first.")

### Total events processed

In [ ]:
total_events = int(event_counts_df["event_count"].sum()) if not event_counts_df.empty else 0
print(f"Total developer_events processed (across closed windows): {total_events}")

### Average salary by country / role / experience range

In [ ]:
def weighted_avg(df, group_col, value_col="avg_salary", weight_col="event_count"):
    if df.empty:
        return pd.Series(dtype=float)
    clean = df.dropna(subset=[value_col])
    return (
        clean.groupby(group_col)
        .apply(lambda g: (g[value_col] * g[weight_col]).sum() / g[weight_col].sum())
        .sort_values(ascending=False)
    )


if not salary_breakdown_df.empty:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    weighted_avg(salary_breakdown_df, "country").plot(kind="bar", ax=axes[0], title="Avg salary by country")
    weighted_avg(salary_breakdown_df, "role").plot(kind="bar", ax=axes[1], title="Avg salary by role")
    weighted_avg(salary_breakdown_df, "experience_range").plot(kind="bar", ax=axes[2], title="Avg salary by experience")
    plt.tight_layout()
    plt.show()
else:
    print("No salary_breakdown records - nothing to plot.")

### Salary distribution

Histogram of per-segment (window x country x role x experience) average salaries - an approximation built from streaming aggregates, not a true per-individual histogram (`salary_analytics` never carries raw per-event salaries).

In [ ]:
if not salary_breakdown_df.empty:
    salary_breakdown_df["avg_salary"].plot(kind="hist", bins=15, title="Distribution of per-segment average salaries")
    plt.xlabel("Average salary ($)")
    plt.show()

### Technology popularity and average salary

In [ ]:
if not technology_counts_df.empty:
    tech_totals = technology_counts_df.groupby("technology")["event_count"].sum().sort_values(ascending=False).head(15)
    tech_salary = weighted_avg(technology_counts_df, "technology").reindex(tech_totals.index)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    tech_totals.plot(kind="bar", ax=axes[0], title="Technology popularity (event count)")
    tech_salary.dropna().plot(kind="bar", ax=axes[1], title="Avg salary by technology")
    plt.tight_layout()
    plt.show()
else:
    print("No technology_counts records - nothing to plot.")

## Part B — Prediction vs. Real Salary

Takes real historical events from `developer_events` (published by `run_dataset_producer.ipynb`) - which carry the REAL `ConvertedCompYearly` from the original survey row - runs them through the trained model directly, and compares predicted vs. real salary.

This scores the model directly with `PipelineModel.transform()` rather than round-tripping through `salary_requests`/`salary_predictions`, so it works even if `run_prediction_stream.ipynb` isn't currently running.

**Caveat:** these events come from the same dataset the model was trained on, so this is a sanity/demo check of individual predictions, not a rigorous held-out evaluation - that's what `models/model_metrics.json`'s test-set RMSE/R² (from the actual train/validation/test split) already is.

In [ ]:
from src.common.schemas import DEVELOPER_EVENT_SCHEMA

raw_events = (
    spark.read.format("kafka")
    .option("kafka.bootstrap.servers", settings.KAFKA_BOOTSTRAP_SERVERS)
    .option("subscribe", settings.KAFKA_DATASET_TOPIC)
    .option("startingOffsets", "earliest")
    .load()
    .select(F.col("value").cast("string").alias("raw_value"))
    .withColumn("event", F.from_json(F.col("raw_value"), DEVELOPER_EVENT_SCHEMA))
    .select("event.*")
)

events_with_salary = raw_events.filter(
    F.col("event_id").isNotNull() & F.col("ConvertedCompYearly").isNotNull()
)
print("Events with a known real salary:", events_with_salary.count())

In [ ]:
from pyspark.ml import PipelineModel

from src.common.spark_utils import reverse_log1p_predictions
from src.streaming.prediction_stream import STRING_FEATURE_COLUMNS, load_model_metadata

SAMPLE_SIZE = 30

sample_events = (
    events_with_salary.fillna("Unknown", subset=STRING_FEATURE_COLUMNS)
    .withColumnRenamed("YearsCodePro", "YearsCodeProNumeric")
    .limit(SAMPLE_SIZE)
)

metadata = load_model_metadata()
model = PipelineModel.load(settings.MODEL_PATH)
print(f"Using model: {metadata['selected_model']} (trained {metadata['trained_at']})")

scored = model.transform(sample_events)
scored = reverse_log1p_predictions(scored, log_prediction_col="log_prediction", output_col="predicted_salary")

comparison = scored.select(
    F.col("event_id"),
    F.col("Country"),
    F.col("DevType"),
    F.col("ConvertedCompYearly").alias("real_salary"),
    F.round(F.col("predicted_salary"), 2).alias("predicted_salary"),
)
comparison_df = comparison.toPandas()
comparison_df["abs_error"] = (comparison_df["predicted_salary"] - comparison_df["real_salary"]).abs()
comparison_df["pct_error"] = comparison_df["abs_error"] / comparison_df["real_salary"] * 100
comparison_df

### Real vs. predicted salary

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(comparison_df["real_salary"], comparison_df["predicted_salary"])
max_value = max(comparison_df["real_salary"].max(), comparison_df["predicted_salary"].max())
plt.plot([0, max_value], [0, max_value], linestyle="--", color="gray", label="Perfect prediction")
plt.xlabel("Real salary ($)")
plt.ylabel("Predicted salary ($)")
plt.title("Real vs. Predicted Salary")
plt.legend()
plt.show()

print("Mean absolute error: $%.2f" % comparison_df["abs_error"].mean())
print("Mean percentage error: %.1f%%" % comparison_df["pct_error"].mean())

### Error distribution

In [ ]:
comparison_df["pct_error"].plot(kind="hist", bins=15, title="Percentage error distribution")
plt.xlabel("Percentage error (%)")
plt.show()